# Assignment 7.1: Music Genre and Composer Classification Using Deep Learning

| <div align="center"> Field </div>| <div align="center"> Details </div> |
|-------|---------|
| **Team Members** | Pros Loung, Nicole Rowley, Reema Eid |
| **Course** | AAI-511 Neural Networks and Deep Learning |
| **FINAL PROJECT** | Music Genre and Composer Classification Using Deep Learning |
| **GitHub Repository** | https://github.com/nicole-rowley-msaai/MS_AAI_511_Neural_Networks_DL |

## Team Contribution Map

| <div align="center"> Team member </div> | <div align="center"> Workstream </div> | <div align="center"> Sections Owned </div> |
|---|---|---|
| **Pros** | | |
| **Nicole** | |  |
| **Reema** |  | |
| **Team** | Final validation, Final Report, and GitHub submission | Notebook 2 final report sections and presentation |

# Introduction

Music is a form of art that is ubiquitous and has a rich history. Different composers have created music with their unique styles and compositions. However, identifying the composer of a particular piece of music can be a challenging task, especially for novice musicians or listeners. The proposed project aims to use deep learning techniques to identify the composer of a given piece of music accurately.

# Objective

The primary objective of this project is to develop a deep learning model that can predict the composer of a given musical score accurately. The project aims to accomplish this objective by using two deep learning techniques: Long Short-Term Memory (LSTM) and Convolutional Neural Network (CNN).

# Methodology

The proposed project will be implemented using the following steps:

1. Data Collection: Use the provided dataset of musical scores and associated composer labels.
2. Data Pre-processing: Convert scores into a model-ready representation (for example MIDI/event sequences) and apply quality checks and augmentation where appropriate.
3. Feature Extraction: Extract relevant musical features such as note patterns, chord progressions, rhythm, and tempo.
4. Model Building: Develop deep learning classifiers using both LSTM and CNN architectures.
5. Model Training: Train the models on preprocessed and feature-engineered data using consistent train/validation/test splits.
6. Model Evaluation: Evaluate performance using accuracy, precision, recall, and F1-score.
7. Model Optimization: Tune hyperparameters and regularization settings to improve generalization.

# Final Project Setup and Pipeline

1. Project Setup
- Define project scope, assumptions, and success criteria.
- Create reproducible environment and dependency list.
- Set random seeds and experiment tracking conventions.

2. Load Data and Exploration
- Load raw music-score data and labels.
- Check class balance, missing labels, and duplicate samples.
- Establish train/validation/test split strategy with stratification.

3. Representation and Preprocessing
- Convert source files to a consistent symbolic representation.
- Normalize sequence length strategy (padding/truncation/windows).
- Build preprocessing pipeline that can be reused for inference.

4. Feature Engineering
- Build two branches of inputs: sequence-based inputs for LSTM and image/grid-like representations for CNN (if applicable).
- Generate optional handcrafted features (tempo, pitch range, interval stats) for comparison baselines.

5. Deep Model Development
- LSTM track: embedding/sequence layers, recurrent blocks, dropout, dense classifier.
- CNN track: convolution blocks, pooling, normalization, dense classifier.
- Keep model definitions modular to support ablation studies.

- **Training and Validation**
    - Use early stopping and model checkpointing.
    - Run controlled experiments across learning rate, batch size, depth, and dropout.
    - Track every run with config, metrics, and notes.

- **Evaluation and Error Analysis**
    - Evaluate best models on hold-out test set.
    - Report accuracy, precision, recall, F1-score, and confusion matrix.
    - Analyze frequent misclassifications and class-specific weaknesses.

6. Model Selection and Optimization
- Compare LSTM vs CNN against objective metrics and training stability.
- Select final model using both metric quality and robustness.
- Apply final hyperparameter tuning pass.

7. Packaging and Reproducibility
- Save preprocessing artifacts, label encoder, and model weights.
- Create an inference function for unseen music input.
- Document end-to-end run steps for reproducibility.

8. Final Deliverables
- Final notebook with narrative and visual results.
- Short report: problem, method, results, limitations, future work.

# Pipeline Connection Diagram

```mermaid
flowchart TD
    A[1. Project Setup] --> B[2. Load Data and Exploration]
    B --> C[3. Representation and Preprocessing]
    C --> D[4. Feature Engineering]
    D --> F[5. Deep Model Development<br/>LSTM and CNN Tracks<br/>Training, Validation, and Error Analysis]
    F --> I[6. Model Selection and Optimization]
    I --> J[7. Packaging and Reproducibility]
    J --> K[8. Final Deliverables]

    F -. Compare architectures .-> I
    F -. Feedback loop for improvements .-> C
    I -. Retune if needed .-> F
```

## 1.0 Project Setup
### Owner: Pros


In [15]:
# Install libraries
# Use the following command to install the pretty_midi library if not already installed
#!pip install pretty_midi
#!pip install kagglehub
#!pip install pygame

In [ ]:
# Load Library 
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import zipfile
import torch
import torch.nn as nn
import pretty_midi

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple
from pathlib import Path
from collections import Counter


## 2. Load Data and Exploration
### Onwer: Pros

- Load raw music-score data and labels.
- Check class balance, missing labels, and duplicate samples.
- Establish train/validation/test split strategy with stratification.

In [17]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("blanderbuss/midi-classic-music")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Loung\.cache\kagglehub\datasets\blanderbuss\midi-classic-music\versions\1


In [44]:
# Load MIDI files directly from midiclassics.zip and create reproducible split

data_root = Path(r"C:\Users\Loung\USD\AAI-551 Neural Networks\MS_AAI_511_Neural_Networks_DL")
zip_path = data_root / "midiclassics.zip"
extract_root = data_root / "midiclassics_extracted"

if not zip_path.exists():
    raise FileNotFoundError(f"Could not find dataset archive: {zip_path}")

if not extract_root.exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_root)

# Only load these composers
target_composers = ["Bach", "Beethoven", "Chopin", "Mozart"]
target_composers_lower = {name.lower() for name in target_composers}

composer_files = []
for composer_dir in extract_root.iterdir():
    if not composer_dir.is_dir() or composer_dir.name.lower() not in target_composers_lower:
        continue

    for midi_file in composer_dir.rglob("*.mid"):
        composer_files.append(
            {
                "path": str(midi_file),
                "composer": composer_dir.name,
            }
        )

if not composer_files:
    raise ValueError("No target composer MIDI files were found in the extracted archive.")

max_total_files = 200
if len(composer_files) > max_total_files:
    sample_labels = [item["composer"].lower() for item in composer_files]
    composer_files, _ = train_test_split(
        composer_files,
        train_size=max_total_files,
        random_state=42,
        stratify=sample_labels,
    )

labels = [item["composer"].lower() for item in composer_files]
train_records, temp_records = train_test_split(
    composer_files,
    test_size=0.30,
    random_state=42,
    stratify=labels,
)
temp_labels = [item["composer"].lower() for item in temp_records]
dev_records, test_records = train_test_split(
    temp_records,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels,
)

split_records = {
    "train": train_records,
    "dev": dev_records,
    "test": test_records,
}
midi_data = {"dev": [], "train": [], "test": []}
load_errors = []

for split, records in split_records.items():
    for record in records:
        midi_file = Path(record["path"])
        try:
            pm = pretty_midi.PrettyMIDI(str(midi_file))
            midi_data[split].append(
                {
                    "path": str(midi_file),
                    "composer": record["composer"],
                    "midi": pm,
                }
            )
        except Exception as e:
            load_errors.append((str(midi_file), str(e)))

print(f"Using dataset archive: {zip_path.name}")
print(f"Extracted files directory: {extract_root}")
print(f"Total sampled MIDI files: {len(composer_files)}")

# Convenience lists if you want to keep your previous variable names
# midi_dev = [item["midi"] for item in midi_data["dev"]]
# midi_train = [item["midi"] for item in midi_data["train"]]
# midi_test = [item["midi"] for item in midi_data["test"]]

# Quick verification output
for split in ["train", "dev", "test"]:
    composer_counts = Counter(item["composer"].lower() for item in midi_data[split])
    print(f"{split.upper()} files loaded: {len(midi_data[split])}")
    for composer_name in target_composers:
        print(f"  - {composer_name}: {composer_counts.get(composer_name.lower(), 0)}")

if load_errors:
    print(f"\nFiles with load errors: {len(load_errors)}")
    for file_path, err in load_errors[:5]:
        print(f"  - {file_path}: {err}")
        

c:\Users\Loung\USD\AAI-530-IoT\AAI-530-IoT-Assignment-4.1\.venv\Lib\site-packages\pretty_midi\pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


Using dataset archive: midiclassics.zip
Extracted files directory: C:\Users\Loung\USD\AAI-551 Neural Networks\MS_AAI_511_Neural_Networks_DL\midiclassics_extracted
Total sampled MIDI files: 200
TRAIN files loaded: 140
  - Bach: 88
  - Beethoven: 18
  - Chopin: 12
  - Mozart: 22
DEV files loaded: 30
  - Bach: 19
  - Beethoven: 4
  - Chopin: 2
  - Mozart: 5
TEST files loaded: 30
  - Bach: 19
  - Beethoven: 4
  - Chopin: 3
  - Mozart: 4


In [45]:
# Convenience lists if you want to keep your previous variable names
midi_dev = [item["midi"] for item in midi_data["dev"]]
midi_train = [item["midi"] for item in midi_data["train"]]
midi_test = [item["midi"] for item in midi_data["test"]]

print ("#" * 75)
print("MIDI Dataset Overview")
print(f"Number of MIDI files in dev set: {len(midi_dev)}")
print(f"Number of MIDI files in train set: {len(midi_train)}")
print(f"Number of MIDI files in test set: {len(midi_test)}")

print ("#" * 75)
# Display the first few MIDI files in each set
print("First few MIDI files in dev set:")
for midi_file in midi_dev[:5]:
    print(midi_file)

print("\nFirst few MIDI files in train set:")
for midi_file in midi_train[:5]:
    print(midi_file)

print("\nFirst few MIDI files in test set:")
for midi_file in midi_test[:5]:
    print(midi_file)

###########################################################################
MIDI Dataset Overview
Number of MIDI files in dev set: 30
Number of MIDI files in train set: 140
Number of MIDI files in test set: 30
###########################################################################
First few MIDI files in dev set:

First few MIDI files in train set:

First few MIDI files in test set:


In [46]:
# Preview the first MIDI file in the dev set as synthesized audio
from IPython.display import Audio, display

sample_rate = 16000
audio_waveform = midi_dev[10].synthesize(fs=sample_rate)
display(Audio(audio_waveform, rate=sample_rate))
print("Displayed an audio preview for the 11th MIDI file in the dev set.")

Displayed an audio preview for the 11th MIDI file in the dev set.


In [47]:
# View split-level shapes for midi_dev, midi_train, and midi_test (memory-safe)

def midi_note_matrix(pm):
    rows = []
    for inst_idx, inst in enumerate(pm.instruments):
        for note in inst.notes:
            rows.append([
                inst_idx,
                note.pitch,
                note.start,
                note.end,
                note.end - note.start,
                note.velocity,
            ])
    return np.array(rows, dtype=float) if rows else np.empty((0, 6))

def summarize_split_shape(split_name, split_data, fs=100):
    print(f"\n{split_name.upper()} split")
    print(f"  list length (files): {len(split_data)}")

    if not split_data:
        print("  first file piano-roll shape: N/A")
        print("  first file note-matrix shape: N/A")
        print("  split stacked piano-roll shape (N, T, 128): (0,)")
        print("  note counts min/mean/max: N/A")
        return

    # Per-file shapes (safe)
    first_roll = split_data[0].get_piano_roll(fs=fs)
    first_notes = midi_note_matrix(split_data[0])
    print(f"  first file piano-roll shape: {first_roll.shape}")
    print(f"  first file note-matrix shape: {first_notes.shape}")

    # Split-level shape stats without allocating padded tensors
    time_steps = [m.get_piano_roll(fs=fs).shape[1] for m in split_data]
    max_t = max(time_steps)
    mean_t = float(np.mean(time_steps))
    print(f"  split stacked piano-roll shape (N, T, 128): ({len(split_data)}, {max_t}, 128) [if padded]")
    print(f"  time steps min/mean/max: {min(time_steps)} / {mean_t:.1f} / {max_t}")

    note_counts = []
    for m in split_data:
        count = sum(len(inst.notes) for inst in m.instruments)
        note_counts.append(count)
    print(f"  note counts min/mean/max: {min(note_counts)} / {np.mean(note_counts):.1f} / {max(note_counts)}")

for split_name, split_data in [("dev", midi_dev), ("train", midi_train), ("test", midi_test)]:
    summarize_split_shape(split_name, split_data, fs=100)


DEV split
  list length (files): 30
  first file piano-roll shape: (128, 19141)
  first file note-matrix shape: (999, 6)
  split stacked piano-roll shape (N, T, 128): (30, 88820, 128) [if padded]
  time steps min/mean/max: 3000 / 18950.4 / 88820
  note counts min/mean/max: 164 / 2471.7 / 17243

TRAIN split
  list length (files): 140
  first file piano-roll shape: (128, 15576)
  first file note-matrix shape: (970, 6)
  split stacked piano-roll shape (N, T, 128): (140, 249996, 128) [if padded]
  time steps min/mean/max: 2000 / 26439.0 / 249996
  note counts min/mean/max: 149 / 3449.8 / 47745

TEST split
  list length (files): 30
  first file piano-roll shape: (128, 3750)
  first file note-matrix shape: (279, 6)
  split stacked piano-roll shape (N, T, 128): (30, 92043, 128) [if padded]
  time steps min/mean/max: 2863 / 19135.2 / 92043
  note counts min/mean/max: 218 / 2003.5 / 10217


## 3. Representation and Preprocessing
### Owner: Pros
- Convert source files to a consistent symbolic representation.
- Normalize sequence length strategy (padding/truncation/windows).
- Build preprocessing pipeline that can be reused for inference.

In [48]:
# Reusable MIDI preprocessing pipeline
@dataclass
class MidiPreprocessorConfig:
    fs: int = 100
    velocity_scale: float = 127.0
    length_percentile: float = 95.0
    window_size: int = 2048
    window_hop: int = 1024
    use_windows: bool = True
    max_files_per_split: int | None = None

class MidiPreprocessor:
    def __init__(self, config: MidiPreprocessorConfig):
        self.config = config
        self.target_length_ = None

    def to_symbolic(self, pm: pretty_midi.PrettyMIDI) -> np.ndarray:
        # Consistent symbolic representation: velocity-normalized piano-roll
        roll = pm.get_piano_roll(fs=self.config.fs).astype(np.float32)
        roll /= self.config.velocity_scale
        return np.clip(roll, 0.0, 1.0)

    def fit(self, midi_list: List[pretty_midi.PrettyMIDI]):
        lengths = [self.to_symbolic(pm).shape[1] for pm in midi_list]
        if not lengths:
            raise ValueError("fit() received an empty MIDI list.")
        self.target_length_ = int(np.percentile(lengths, self.config.length_percentile))
        self.target_length_ = max(self.target_length_, 1)
        return self

    def _pad_or_truncate(self, seq: np.ndarray) -> np.ndarray:
        if self.target_length_ is None:
            raise RuntimeError("Call fit() before transform()")

        t = seq.shape[1]
        if t < self.target_length_:
            pad = np.zeros((128, self.target_length_ - t), dtype=np.float32)
            seq = np.concatenate([seq, pad], axis=1)
        elif t > self.target_length_:
            seq = seq[:, :self.target_length_]
        return seq

    def _window(self, seq: np.ndarray) -> np.ndarray:
        ws = self.config.window_size
        hop = self.config.window_hop
        if ws <= 0 or hop <= 0:
            raise ValueError("window_size and window_hop must be > 0")

        if seq.shape[1] < ws:
            pad = np.zeros((128, ws - seq.shape[1]), dtype=np.float32)
            seq = np.concatenate([seq, pad], axis=1)

        windows = []
        for start in range(0, seq.shape[1] - ws + 1, hop):
            windows.append(seq[:, start:start + ws])

        if not windows:
            windows.append(seq[:, :ws])
        return np.stack(windows, axis=0).astype(np.float32)

    def transform_list(self, midi_list: List[pretty_midi.PrettyMIDI]) -> np.ndarray:
        arrays = []
        for pm in midi_list:
            seq = self.to_symbolic(pm)
            seq = self._pad_or_truncate(seq)
            if self.config.use_windows:
                seq = self._window(seq)  # (num_windows, 128, window_size)
                arrays.append(seq)
            else:
                arrays.append(seq[np.newaxis, ...])  # (1, 128, T)

        if not arrays:
            return np.empty((0, 128, self.config.window_size), dtype=np.float32)

        return np.concatenate(arrays, axis=0).astype(np.float32)

    def transform_path(self, midi_path: str) -> np.ndarray:
        # Inference entrypoint for one unseen MIDI file
        pm = pretty_midi.PrettyMIDI(midi_path)
        return self.transform_list([pm])

In [49]:
# Fit on train split and preprocess in memory-safe chunks to reusable files

config = MidiPreprocessorConfig(
    fs=100,
    length_percentile=95.0,
    window_size=1024,
    window_hop=512,
    use_windows=True,
    max_files_per_split=120,  # increase gradually if needed
    )

preprocessor = MidiPreprocessor(config)
preprocessor.fit(midi_train)

def maybe_limit(split_list, max_files):
    if max_files is None:
        return split_list
    return split_list[:max_files]

split_inputs = {
    "train": maybe_limit(midi_train, config.max_files_per_split),
    "dev": maybe_limit(midi_dev, config.max_files_per_split),
    "test": maybe_limit(midi_test, config.max_files_per_split),
}

artifact_dir = Path("artifacts/preprocessed")
artifact_dir.mkdir(parents=True, exist_ok=True)

def estimate_total_windows(midi_list, fs, target_length, ws, hop):
    total = 0
    for pm in midi_list:
        t = pm.get_piano_roll(fs=fs).shape[1]
        t_eff = max(t, target_length)
        if t_eff < ws:
            t_eff = ws
        n = max(1, 1 + (t_eff - ws) // hop)
        total += n
    return total

def build_split_memmap(split_name, midi_list, batch_size=8):
    total_windows = estimate_total_windows(
        midi_list,
        fs=config.fs,
        target_length=preprocessor.target_length_,
        ws=config.window_size,
        hop=config.window_hop,
    )

    out_path = artifact_dir / f"X_{split_name}.npy"
    X_mm = np.lib.format.open_memmap(
        out_path,
        mode="w+",
        dtype=np.float32,
        shape=(total_windows, 128, config.window_size),
    )

    cursor = 0
    for i in range(0, len(midi_list), batch_size):
        chunk = midi_list[i:i + batch_size]
        arr = preprocessor.transform_list(chunk)
        n = arr.shape[0]
        X_mm[cursor:cursor + n] = arr
        cursor += n

    # Flush to disk and reopen read-only memory map
    del X_mm
    X_mm = np.load(out_path, mmap_mode="r")
    return X_mm, out_path

X_splits = {}
for name, data in split_inputs.items():
    X_mm, out_path = build_split_memmap(name, data, batch_size=6)
    X_splits[name] = X_mm
    print(f"X_{name}.shape = {X_mm.shape}")
    print(f"Saved: {out_path}")

print("Preprocessing complete")
print(f"Target sequence length (for pad/truncate): {preprocessor.target_length_}")

# Inference-ready transform for one unseen MIDI file
example_path = midi_data["dev"][0]["path"]
X_infer = preprocessor.transform_path(example_path)
print(f"X_infer.shape = {X_infer.shape}")
print(f"Example inference file: {example_path}")

X_train.shape = (20337, 128, 1024)
Saved: artifacts\preprocessed\X_train.npy
X_dev.shape = (4957, 128, 1024)
Saved: artifacts\preprocessed\X_dev.npy
X_test.shape = (4963, 128, 1024)
Saved: artifacts\preprocessed\X_test.npy
Preprocessing complete
Target sequence length (for pad/truncate): 85462
X_infer.shape = (165, 128, 1024)
Example inference file: C:\Users\Loung\USD\AAI-551 Neural Networks\MS_AAI_511_Neural_Networks_DL\midiclassics_extracted\Mozart\Viennese Sonatinas K439b n6 2mov.mid


## 4. Feature Engineering
### Owner: Pros
- Build two branches of inputs: sequence-based inputs for LSTM and image/grid-like representations for CNN (if applicable).
- Generate optional handcrafted features (tempo, pitch range, interval stats) for comparison baselines.

In [50]:
# 4. Feature Engineering: LSTM branch, CNN branch, and handcrafted features
from sklearn.preprocessing import LabelEncoder

# Keep this modest for notebook memory; set to None to use all files in each split.
max_files_per_split_feature = 80

def limit_records(records, n):
    if n is None:
        return records
    return records[:n]

def symbolic_from_pm(pm, fs=100):
    roll = pm.get_piano_roll(fs=fs).astype(np.float32) / 127.0
    return np.clip(roll, 0.0, 1.0)

def pad_or_truncate(roll, target_length):
    t = roll.shape[1]
    if t < target_length:
        pad = np.zeros((128, target_length - t), dtype=np.float32)
        return np.concatenate([roll, pad], axis=1)
    return roll[:, :target_length]

def window_roll(roll, window_size=1024, hop=512):
    if roll.shape[1] < window_size:
        pad = np.zeros((128, window_size - roll.shape[1]), dtype=np.float32)
        roll = np.concatenate([roll, pad], axis=1)

    windows = []
    for start in range(0, roll.shape[1] - window_size + 1, hop):
        windows.append(roll[:, start:start + window_size])

    if not windows:
        windows.append(roll[:, :window_size])
    return np.stack(windows, axis=0).astype(np.float32)

def extract_handcrafted_features(pm):
    notes = []
    for inst in pm.instruments:
        notes.extend(inst.notes)

    if not notes:
        return np.array([0.0] * 8, dtype=np.float32)

    notes = sorted(notes, key=lambda n: n.start)
    pitches = np.array([n.pitch for n in notes], dtype=np.float32)
    starts = np.array([n.start for n in notes], dtype=np.float32)
    ends = np.array([n.end for n in notes], dtype=np.float32)
    durations = np.maximum(ends - starts, 1e-6)
    velocities = np.array([n.velocity for n in notes], dtype=np.float32)

    # Robust tempo estimate: fallback to 0.0 if estimation fails
    try:
        tempo = float(pm.estimate_tempo())
    except Exception:
        tempo = 0.0

    pitch_min = float(np.min(pitches))
    pitch_max = float(np.max(pitches))
    pitch_range = pitch_max - pitch_min

    if len(pitches) > 1:
        intervals = np.diff(pitches)
        interval_mean = float(np.mean(intervals))
        interval_std = float(np.std(intervals))
    else:
        interval_mean = 0.0
        interval_std = 0.0

    piece_duration = max(float(pm.get_end_time()), 1e-6)
    notes_per_sec = float(len(notes) / piece_duration)
    mean_duration = float(np.mean(durations))
    mean_velocity = float(np.mean(velocities))

    return np.array(
        [
            tempo,
            pitch_range,
            interval_mean,
            interval_std,
            notes_per_sec,
            mean_duration,
            mean_velocity,
            piece_duration,
        ],
        dtype=np.float32,
    )

def build_split_branches(split_records, target_length, window_size=1024, hop=512, fs=100):
    lstm_samples = []
    cnn_samples = []
    hand_features = []
    labels = []

    for item in split_records:
        pm = item["midi"]
        label = item["composer"]

        roll = symbolic_from_pm(pm, fs=fs)
        roll = pad_or_truncate(roll, target_length)
        roll_windows = window_roll(roll, window_size=window_size, hop=hop)  # (W, 128, window_size)

        # LSTM branch: time-major sequences (W, T, 128)
        lstm_w = np.transpose(roll_windows, (0, 2, 1)).astype(np.float32)

        # CNN branch: image/grid representation with channel dim (W, 128, T, 1)
        cnn_w = roll_windows[..., np.newaxis].astype(np.float32)

        feats = extract_handcrafted_features(pm)

        # Repeat file-level label and handcrafted features for each generated window
        w = roll_windows.shape[0]
        lstm_samples.append(lstm_w)
        cnn_samples.append(cnn_w)
        hand_features.append(np.repeat(feats[np.newaxis, :], w, axis=0))
        labels.extend([label] * w)

    if not lstm_samples:
        return (
            np.empty((0, window_size, 128), dtype=np.float32),
            np.empty((0, 128, window_size, 1), dtype=np.float32),
            np.empty((0, 8), dtype=np.float32),
            np.array([], dtype=object),
        )

    X_lstm = np.concatenate(lstm_samples, axis=0)
    X_cnn = np.concatenate(cnn_samples, axis=0)
    X_feat = np.concatenate(hand_features, axis=0)
    y = np.array(labels, dtype=object)
    return X_lstm, X_cnn, X_feat, y

# Reuse the fitted preprocessing target length from Section 3 if available; fallback to 95th percentile on train
if "preprocessor" in globals() and getattr(preprocessor, "target_length_", None) is not None:
    target_len = preprocessor.target_length_
else:
    train_lengths = [m.get_piano_roll(fs=100).shape[1] for m in midi_train]
    target_len = int(np.percentile(train_lengths, 95))
    target_len = max(target_len, 1)

window_size = 1024
window_hop = 512

feature_records = {
    "train": limit_records(midi_data["train"], max_files_per_split_feature),
    "dev": limit_records(midi_data["dev"], max_files_per_split_feature),
    "test": limit_records(midi_data["test"], max_files_per_split_feature),
}

branch_data = {}
for split_name, records in feature_records.items():
    X_lstm, X_cnn, X_feat, y_text = build_split_branches(
        records,
        target_length=target_len,
        window_size=window_size,
        hop=window_hop,
        fs=100,
    )
    branch_data[split_name] = {
        "X_lstm": X_lstm,
        "X_cnn": X_cnn,
        "X_feat": X_feat,
        "y_text": y_text,
    }

# Shared label encoder for both branches and baseline
label_encoder = LabelEncoder()
all_train_labels = branch_data["train"]["y_text"]
y_train = label_encoder.fit_transform(all_train_labels)
y_dev = label_encoder.transform(branch_data["dev"]["y_text"])
y_test = label_encoder.transform(branch_data["test"]["y_text"])

# Expose final tensors for later model sections
X_lstm_train, X_lstm_dev, X_lstm_test = (
    branch_data["train"]["X_lstm"],
    branch_data["dev"]["X_lstm"],
    branch_data["test"]["X_lstm"],
)
X_cnn_train, X_cnn_dev, X_cnn_test = (
    branch_data["train"]["X_cnn"],
    branch_data["dev"]["X_cnn"],
    branch_data["test"]["X_cnn"],
)
X_feat_train, X_feat_dev, X_feat_test = (
    branch_data["train"]["X_feat"],
    branch_data["dev"]["X_feat"],
    branch_data["test"]["X_feat"],
)

print("Feature engineering complete")
print(f"target_len={target_len}, window_size={window_size}, hop={window_hop}")

print("#" * 75)
print("\nLSTM branch shapes (N, T, F):")
print("  train:", X_lstm_train.shape, "dev:", X_lstm_dev.shape, "test:", X_lstm_test.shape)
print("#" * 75)
print("CNN branch shapes (N, H, W, C):")
print("  train:", X_cnn_train.shape, "dev:", X_cnn_dev.shape, "test:", X_cnn_test.shape)
print("#" * 75)
print("Handcrafted baseline shapes (N, D):")
print("  train:", X_feat_train.shape, "dev:", X_feat_dev.shape, "test:", X_feat_test.shape)
print("#" * 75)
print("Encoded labels:", len(label_encoder.classes_), "classes ->", list(label_encoder.classes_))

Feature engineering complete
target_len=85462, window_size=1024, hop=512
###########################################################################

LSTM branch shapes (N, T, F):
  train: (13200, 1024, 128) dev: (4950, 1024, 128) test: (4950, 1024, 128)
###########################################################################
CNN branch shapes (N, H, W, C):
  train: (13200, 128, 1024, 1) dev: (4950, 128, 1024, 1) test: (4950, 128, 1024, 1)
###########################################################################
Handcrafted baseline shapes (N, D):
  train: (13200, 8) dev: (4950, 8) test: (4950, 8)
###########################################################################
Encoded labels: 4 classes -> ['Bach', 'Beethoven', 'Chopin', 'Mozart']


## 5. Models Development

### 5.1 Simple baseline model for comparison with deep learning model (LSTM and CNN)
### Owner: Pros


### 5.2. LSTM Model Development
### Owner: Reema

In [52]:
# Enter your code here to define the LSTM model architecture, including embedding/sequence layers, recurrent blocks, dropout, and a dense classifier. Make sure to keep the model definitions modular to facilitate ablation studies.

# LSTM model
# Input: X_lstm_train, X_lstm_dev, X_lstm_test
# Shape: (N, T, F) where T=1024 and F=128
# Labels: y_train, y_dev, y_test

### Training, Optimization, and Validation

In [53]:

# Use early stopping and model checkpointing.
# Run controlled experiments across learning rate, batch size, depth, and dropout.
# Track every run with config, metrics, and notes.

### Evaluation and Error Analysis

In [55]:
# Evaluate best models on hold-out test set.
# Report accuracy, precision, recall, F1-score, and confusion matrix.
# Analyze frequent misclassifications and class-specific weaknesses.

### 5.3. CNN Model Development
### Owner: Nicole

In [ ]:
# Enter your code here to define the CNN model architecture, including convolutional blocks, pooling layers, normalization, and a dense classifier. Make sure to keep the model definitions modular to facilitate ablation studies.

# CNN model
# Input: X_cnn_train, X_cnn_dev, X_cnn_test
# Shape: (N, H, W, C) = (N, 128, 1024, 1)
# Labels: y_train, y_dev, y_test

###  Training, Optimization and Validation

In [26]:
# Use early stopping and model checkpointing.
# Run controlled experiments across learning rate, batch size, depth, and dropout.
# Track every run with config, metrics, and notes.

### Evaluation and Error Analysis

In [27]:
# Evaluate best models on hold-out test set.
# Report accuracy, precision, recall, F1-score, and confusion matrix.
# Analyze frequent misclassifications and class-specific weaknesses.

## 6. Model Selection and Optimization
### Owner: Reema and Nicole
- Compare LSTM vs CNN against objective metrics and training stability.
- Select final model using both metric quality and robustness.
- Apply final hyperparameter tuning pass.

## 7. Packaging and Reproducibility
### Owner: Reema or Nicole depending on which model is selected.

In [28]:
# Save preprocessing artifacts, label encoder, and model weights.
# Create an inference function for unseen music input.
# Document end-to-end run steps for reproducibility.

### Optional demo cell: predict composer for a sample input.

## 8. Final Deliverables
### Owner: Pros
- Final notebook with narrative and visual results.

### Owner: Team
- Final Report: problem, method, results, limitations, future work.
